# RBC instance segmentation — yolo11n-seg with a higher-resolution mask head

Same dataset and pretrained backbone as before, but with a custom `ProtoHR` mask module that adds one extra upsampling stage — the model's actual output grid becomes `imgsz/2` instead of the default `imgsz/4`. This is a real architecture change (verified via a CPU smoke test: proto resolution doubled 160x160 -> 320x320 at test imgsz=640, gradients flow correctly into the two new layers), not a training-time-only trick like `mask_ratio` turned out to be.

**Only run this after `rbc_seg_maskratio_imgz1024` finishes** — the GPU can only fit one training job at a time on this card.

In [ ]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.nn.modules.conv import Conv

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## The custom mask head

`ProtoHR` is the same as the stock `Proto` module, with one extra `upsample -> conv` stage appended. Every pretrained weight that still fits the new shapes gets transplanted in (`cv1`, `cv2`, `upsample1`, `cv4`) — only the two brand-new layers (`upsample2`, `cv3`) start randomly initialized.

In [ ]:
class ProtoHR(nn.Module):
    """Proto with an extra upsampling stage: outputs masks at imgsz/2 instead of imgsz/4."""

    def __init__(self, c1: int, c_: int = 256, c2: int = 32):
        super().__init__()
        self.cv1 = Conv(c1, c_, k=3)
        self.upsample1 = nn.ConvTranspose2d(c_, c_, 2, 2, 0, bias=True)
        self.cv2 = Conv(c_, c_, k=3)
        self.upsample2 = nn.ConvTranspose2d(c_, c_, 2, 2, 0, bias=True)  # the new stage
        self.cv3 = Conv(c_, c_, k=3)                                     # refines after the new upsample
        self.cv4 = Conv(c_, c2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.cv1(x)
        x = self.upsample1(x)
        x = self.cv2(x)
        x = self.upsample2(x)
        x = self.cv3(x)
        return self.cv4(x)


def build_model() -> YOLO:
    model = YOLO("yolo11n-seg.pt")
    segment_head = model.model.model[-1]
    old_proto = segment_head.proto

    c1 = old_proto.cv1.conv.in_channels
    c_ = old_proto.cv1.conv.out_channels
    c2 = old_proto.cv3.conv.out_channels

    new_proto = ProtoHR(c1, c_, c2)
    new_proto.cv1.load_state_dict(old_proto.cv1.state_dict())
    new_proto.cv2.load_state_dict(old_proto.cv2.state_dict())
    new_proto.upsample1.load_state_dict(old_proto.upsample.state_dict())
    new_proto.cv4.load_state_dict(old_proto.cv3.state_dict())

    segment_head.proto = new_proto
    return model

## Sanity-check the swap before spending GPU time on it

Quick CPU forward + backward pass — confirms the proto output is really doubled and gradients reach the new layers.

In [ ]:
_check_model = build_model()
_check_model.model.train()
dummy = torch.zeros(1, 3, 640, 640)
out = _check_model.model(dummy)
proto_tensor = out["proto"] if isinstance(out, dict) else out[1]["proto"]
print("proto output shape:", proto_tensor.shape, "(expect 320x320 at test imgsz=640, was 160x160 before)")

loss = proto_tensor.sum()
loss.backward()
new_proto_module = _check_model.model.model[-1].proto
grad_ok = new_proto_module.upsample2.weight.grad is not None and new_proto_module.cv3.conv.weight.grad is not None
print("gradients flow into new layers:", grad_ok)
del _check_model, dummy, out, proto_tensor, loss

## Config

In [ ]:
from pathlib import Path

DATASET_YAML = Path(r"F:\Livo\Data - 2026\Rbc\yolo_seg_dataset\dataset.yaml")
RUNS_DIR = Path(r"F:\Livo\Data - 2026\Rbc\runs")
RUN_NAME = "rbc_seg_protohr_imgz768"

EPOCHS = 15
IMG_SIZE = 768    # was 1024 -- that gave a 512x512 proto (4x the standard run's 256x256), projected 5+ hours per
                   # epoch even at batch=1 (already the floor). 768 -> 384x384 proto: still finer than the
                   # standard imgsz=1024 run's 256x256, at a more tractable memory/speed cost.
BATCH = 1
PATIENCE = 0
MASK_RATIO = 2      # matched to the new imgsz/2 output resolution regardless of imgsz (was 4, matched to imgsz/4)

print(DATASET_YAML.read_text())

## Train

In [ ]:
model = build_model()

results = model.train(
    data=str(DATASET_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    workers=0,      # Windows + Jupyter: multiprocessing DataLoader workers deadlock at epoch start, this avoids it
    plots=False,    # label/results plotting chokes on ~712k instances (170/img avg) -- big RAM spike, skip it
    mosaic=0.0,     # mosaic stitches 4 full images together before cropping -- momentary 4x memory, disable on 8GB
    mask_ratio=MASK_RATIO,
)

## Validate the best checkpoint

In [ ]:
from ultralytics import YOLO as _YOLO

best_weights = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
best_model = _YOLO(str(best_weights))
metrics = best_model.val(data=str(DATASET_YAML))

print("mask mAP50-95:", metrics.seg.map)
print("mask mAP50:   ", metrics.seg.map50)

## Quick visual check — run inference on one val image and draw the predicted contours

In [ ]:
import cv2
import numpy as np

val_dir = DATASET_YAML.parent / "images" / "val"
sample_path = sorted(val_dir.glob("*.jpg"))[0]

result = best_model.predict(str(sample_path), conf=0.25)[0]

img = cv2.imread(str(sample_path))
for contour in result.masks.xy:
    pts = contour.astype(np.int32).reshape(-1, 1, 2)
    cv2.polylines(img, [pts], True, (0, 255, 0), 1, cv2.LINE_AA)

out_path = DATASET_YAML.parent / "sample_prediction_protohr.jpg"
cv2.imwrite(str(out_path), img)
print(f"{len(result.masks.xy)} cells detected -> {out_path}")

## Next steps

- `runs/rbc_seg_protohr_imgz1024/weights/best.pt` is the file to load for real inference.
- Compare its mask mAP50-95 and the visual contour smoothness against `runs/rbc_seg_maskratio_imgz1024` (the standard-architecture run) on the same held-out test images before deciding which model to keep.
- If GPU memory overflows at `batch=1`, the next lever is dropping `imgsz` (e.g. 768), not batch (already at the floor).